In [2]:
from dataclasses import dataclass, field, replace
from functools import reduce
from typing import Callable


class PermissionError(Exception):
    pass

@dataclass(slots=True)
class User:
    name: str
    is_active: bool
    roles: set[str]
    has_mfa: bool
    subscription_tier: str

@dataclass(slots=True)
class Request:
    path: str
    action: str
    requires_audit: bool = False
    required_role: str | None = None
    audit_log: list[str] = field(default_factory=list[str])
    access_granted: bool = False


type PolicyFn = Callable[[User, Request], Request]

def active_user(user: User, request: Request) -> Request:
    if not user.is_active:
        raise PermissionError("Inactive users cannot make requests")
    return request

def mfa_required(user: User, request: Request) -> Request:
    if request.action == "delete" and not user.has_mfa:
        raise PermissionError("MFA is required for delete actions")   
    return request

def role_required(user: User, request: Request) -> Request:
    if request.required_role and request.required_role not in user.roles:
        raise PermissionError(f"Missing required role: {request.required_role}")
    return request

def audit(user: User, request: Request) -> Request:
    if not request.requires_audit:
        return request

    return replace(
        request,
        audit_log=request.audit_log
        + [f"{user.name} performed {request.action} on {request.path}"],
    )

def grant_access(user: User, request: Request) -> Request:
    return replace(request, access_granted=True)

def apply_policies(user: User, request: Request, policies: list[PolicyFn]) -> Request:
    return reduce(lambda current, policy: policy(user, current), policies, request)



# def process_request(user: User, request: Request) -> Request:
#     if not user.is_active:
#         raise PermissionError("Inactive users cannot make requests")

#     if request.action == "delete" and not user.has_mfa:
#         raise PermissionError("MFA is required for delete actions")

#     if request.required_role and request.required_role not in user.roles:
#         raise PermissionError(f"Missing required role: {request.required_role}")

#     if request.requireds_audit:
#         request.audit_log.append(
#             f"{user.name} performed {request.action} on {request.path}"
#         )

#     request.access_granted = True
#     return request

def main() -> None:
    user = User(
        name="Arjan",
        is_active=True,
        roles={"admin"},
        has_mfa=True,
        subscription_tier="pro",
    )

    request = Request(
        path="/admin/users",
        action="delete",
        requires_audit=True,
        required_role="admin",
    )

    policies = [
        active_user,
        mfa_required,
        role_required,
        audit,
        grant_access,
    ]
    request = apply_policies(user, request, policies)
    # policy = CompositePolicy(
    #     [
    #         ActiveUserPolicy(),
    #         MfaRequiredPolicy(),
    #         RoleRequiredPolicy(),
    #         AuditPolicy(),
    #         GrantAccessPolicy(),
    #     ],
    # )

    # policy.apply(user, request)
    print(request)

if __name__ == "__main__":
    main()

Request(path='/admin/users', action='delete', requires_audit=True, required_role='admin', audit_log=['Arjan performed delete on /admin/users'], access_granted=True)
